# 03 - Champion Selection & Experiment Balance Check

This notebook:
1. Dynamically queries `db.sqlite3` for all completed LLaMEA experiments present in the database.
2. Displays an **experiment summary** grouped by problem ID, dimension, noise level, prompt strategy, and LLM model family.
3. Selects separate **Clean** ($\sigma=0.0$) and **Noisy** ($\sigma>0.0$) champion algorithms per condition (lowest ground-truth error across iterations, with evaluation efficiency as tie-breaker).
4. Exports `data/champions.json` ready for multi-run empirical evaluation in **Notebook 04** (`04_evaluate_champions.ipynb`).


In [4]:
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Add src to path
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from core.config import DATA_DIR, PROJECT_ROOT
from infra.storage import get_db_connection, get_db_engine

CHAMPIONS_PATH = DATA_DIR / 'champions.json'
print(f'Champions Output Path: {CHAMPIONS_PATH}')


Champions Output Path: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/data/champions.json


## 1. Experiment Balance Check

In [5]:
# Query all completed experiments dynamically
query_exps = """
SELECT 
    id as exp_id,
    problem_id,
    dim,
    noise_std,
    prompt_strategy,
    llm_name,
    status,
    best_algorithm,
    best_final_error
FROM experiments
WHERE status = 'completed'
ORDER BY problem_id, dim, noise_std, prompt_strategy, exp_id
"""
with get_db_connection() as conn:
    df_exps = pd.read_sql_query(query_exps, conn)

if df_exps.empty:
    raise RuntimeError("No completed experiments found in database. Please run Notebook 02 first.")

print(f'Total completed experiments in database: {len(df_exps)}')
print('\n=== Completed Experiments Summary (Problem x Dim x Noise x Strategy) ===')
summary = df_exps.groupby(['problem_id', 'dim', 'noise_std', 'prompt_strategy']).size().reset_index(name='runs_count')
display(summary) if 'display' in globals() else print(summary.to_string(index=False))


Total completed experiments in database: 292

=== Completed Experiments Summary (Problem x Dim x Noise x Strategy) ===
 problem_id  dim  noise_std prompt_strategy  runs_count
          1    2       0.00        baseline           3
          1    2       0.00          guided           2
          1    2       0.00        thinking           2
          1    2       0.00   vectorization           2
          1    2       0.05        baseline           2
          1    2       0.05          guided           5
          1    2       0.05        thinking           2
          1    2       0.05   vectorization           2
          1    3       0.00        baseline           3
          1    3       0.00          guided           5
          1    3       0.00        thinking           2
          1    3       0.00   vectorization           2
          1    3       0.05        baseline           3
          1    3       0.05          guided           3
          1    3       0.05        thinki

## 2. Select Problem-Specific Champions (Dynamically)

In [6]:
# Query all iterations from completed experiments to find true lowest final_error per (llm_name, problem, dim, mode, strategy)
iter_query = '''
SELECT 
    e.problem_id,
    e.dim,
    e.noise_std,
    e.prompt_strategy,
    e.id as experiment_id,
    e.llm_name,
    i.id as iteration_id,
    i.algorithm_name,
    i.final_error,
    i.evaluations_used,
    i.code_path
FROM iterations i
JOIN experiments e ON i.experiment_id = e.id
WHERE e.status = 'completed'
  AND i.final_error IS NOT NULL
ORDER BY e.llm_name, e.problem_id, e.dim, e.noise_std, e.prompt_strategy, i.final_error ASC, i.evaluations_used ASC, i.id ASC
'''
with get_db_connection() as conn:
    df_iters = pd.read_sql_query(iter_query, conn)

if df_iters.empty:
    raise RuntimeError('No completed iterations found in database.')

champions = {}
print('=== Dimension-Specific Champions (Dynamically Discovered from DB) ===')

for llm_name in sorted(df_iters['llm_name'].dropna().unique()):
    champions[llm_name] = {}
    m_df = df_iters[df_iters['llm_name'] == llm_name]
    print(f'\n🔹 Model: {llm_name}')
    
    for (p_id, dim, strat), group in m_df.groupby(['problem_id', 'dim', 'prompt_strategy']):
        # 1. Clean Champion (noise_std == 0.0)
        clean_grp = group[group['noise_std'] == 0.0]
        if not clean_grp.empty:
            best_clean = clean_grp.iloc[0]
            k_clean = f'f{p_id}_{dim}D_clean_{strat}'
            algo_n = best_clean['algorithm_name']
            exp_i = best_clean['experiment_id']
            f_err = best_clean['final_error']
            champions[llm_name][k_clean] = {
                'problem_id': int(p_id),
                'dim': int(dim),
                'mode': 'clean',
                'noise_std': 0.0,
                'prompt_strategy': str(strat),
                'experiment_id': int(exp_i),
                'iteration_id': int(best_clean['iteration_id']),
                'algorithm_name': str(algo_n),
                'final_error': float(f_err),
                'evaluations_used': int(best_clean['evaluations_used']) if pd.notnull(best_clean['evaluations_used']) else None,
                'code_path': str(best_clean['code_path']),
                'llm_name': str(llm_name),
            }
            print(f'  f{p_id} {dim}D Clean [{strat:<13}]: {algo_n} (Exp #{exp_i}) -> err = {f_err:.6e}')
        
        # 2. Noisy Champion (noise_std > 0.0)
        noisy_grp = group[group['noise_std'] > 0.0]
        if not noisy_grp.empty:
            best_noisy = noisy_grp.iloc[0]
            k_noisy = f'f{p_id}_{dim}D_noisy_{strat}'
            algo_n = best_noisy['algorithm_name']
            exp_i = best_noisy['experiment_id']
            f_err = best_noisy['final_error']
            n_std = best_noisy['noise_std']
            champions[llm_name][k_noisy] = {
                'problem_id': int(p_id),
                'dim': int(dim),
                'mode': 'noisy',
                'noise_std': float(n_std),
                'prompt_strategy': str(strat),
                'experiment_id': int(exp_i),
                'iteration_id': int(best_noisy['iteration_id']),
                'algorithm_name': str(algo_n),
                'final_error': float(f_err),
                'evaluations_used': int(best_noisy['evaluations_used']) if pd.notnull(best_noisy['evaluations_used']) else None,
                'code_path': str(best_noisy['code_path']),
                'llm_name': str(llm_name),
            }
            print(f'  f{p_id} {dim}D Noisy [{strat:<13}]: {algo_n} (Exp #{exp_i}, std={n_std}) -> err = {f_err:.6e}')

# Save champions.json
CHAMPIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(CHAMPIONS_PATH, 'w', encoding='utf-8') as f:
    json.dump(champions, f, indent=2)

total_champs = sum(len(v) for v in champions.values())
print(f'\n✨ Exported {total_champs} dimension-specific champion(s) across {len(champions)} DB model(s) to {CHAMPIONS_PATH}')


=== Dimension-Specific Champions (Dynamically Discovered from DB) ===

🔹 Model: qwen2.5-coder-14b-instruct-q4_k_m.gguf
  f1 2D Clean [baseline     ]: GradientDescentOptimizer (Exp #43) -> err = 1.278977e-13
  f1 2D Noisy [baseline     ]: EnhancedNoiseResilientOptimizer (Exp #68, std=0.05) -> err = 5.233681e-02
  f1 2D Clean [guided       ]: DEOptimizer (Exp #63) -> err = 0.000000e+00
  f1 2D Noisy [guided       ]: NoisyOptimizationAlgorithm (Exp #82, std=0.05) -> err = 2.330717e-02
  f1 2D Clean [thinking     ]: EnhancedExplorationExploitationOptimizer (Exp #51) -> err = 1.671822e-09
  f1 2D Noisy [thinking     ]: ImprovedNoisyOptimizationAlgorithm (Exp #72, std=0.05) -> err = 4.971105e-03
  f1 2D Clean [vectorization]: AdvancedGradientOptimizer (Exp #59) -> err = 0.000000e+00
  f1 2D Noisy [vectorization]: ImprovedNoisyOptimizationSolver (Exp #77, std=0.05) -> err = 7.218205e-02
  f1 3D Clean [baseline     ]: EnhancedDifferentialEvolutionWithLocalSearch (Exp #34) -> err = 0.000000e+00